# 🏆 AIGOAT Task 2 — Final Optimized Solution

## Critical Fix: Train with Evaluation-Aligned Loss

**Problem**: Evaluation uses per-sample min-max normalization, but we trained with raw BerHu.

**Solution**: Use **Normalized MSE Loss** that matches the evaluation metric exactly!

### Changes from v1.53:
1. ✅ **Normalized MSE Loss** - matches evaluation RMSE exactly
2. ✅ **Multi-scale loss** - supervise at 448, 224, 112
3. ✅ **Edge-aware gradient loss** - better depth discontinuities
4. ✅ **100 epochs** - more convergence
5. ✅ **Lower distillation** - focus on GT alignment

### Target:
- RMSE: **< 0.12** (from 0.135)
- Score: **> 1.7**

In [2]:
!nvidia-smi

Sun Feb  8 06:43:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   32C    P0            121W /  700W |   11023MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
%%capture
!pip install onnx onnxruntime-gpu timm albumentations transformers accelerate

In [4]:
import os, gc, math, random, time
import numpy as np
import cv2
from tqdm import tqdm
from collections import defaultdict
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import timm
import onnx
import onnxruntime as ort
import albumentations as A
import matplotlib.pyplot as plt

DEVICE = 'cuda'
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

GPU: NVIDIA H100 80GB HBM3
VRAM: 85.0 GB


In [5]:
CFG = {
    'train_root': '/kaggle/input/ai-goat-1-0-task2-dataset/train',
    'img_size': 448,
    
    # Tiny model
    'encoder_name': 'mobilenetv3_small_100',
    'decoder_channels': 48,
    'dropout': 0.1,
    
    # Longer training
    'epochs': 100,
    'batch_size': 32,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    
    # Lower distillation - focus on GT
    'w_distill_start': 0.3,
    'w_distill_end': 0.02,
    
    # Teacher
    'teacher_model': 'depth-anything/Depth-Anything-V2-Large-hf',
    
    'onnx_path': 'depth_final.onnx',
}
print("Config loaded ✓")

Config loaded ✓


---
## 1 · Data

In [6]:
all_files = os.listdir(CFG['train_root'])
all_ids = sorted(set(
    f.replace('_image.png', '').replace('_depth_map.png', '')
    for f in all_files if f.endswith('.png')
))
print(f"Total: {len(all_ids)} samples")

random.shuffle(all_ids)
val_ids = all_ids[:50]
train_ids = all_ids  # ALL

Total: 9200 samples


---
## 2 · Teacher

In [7]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

print(f"Loading: {CFG['teacher_model']}")
teacher_proc = AutoImageProcessor.from_pretrained(CFG['teacher_model'])
teacher = AutoModelForDepthEstimation.from_pretrained(
    CFG['teacher_model'], torch_dtype=torch.float16
).to(DEVICE).eval()
print(f"Teacher: {sum(p.numel() for p in teacher.parameters())/1e6:.0f}M params")

2026-02-08 06:43:44.267236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770533024.282182   15140 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770533024.286518   15140 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770533024.297669   15140 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770533024.297685   15140 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770533024.297687   15140 computation_placer.cc:177] computation placer alr

Loading: depth-anything/Depth-Anything-V2-Large-hf
Teacher: 335M params


In [8]:
@torch.no_grad()
def gen_teacher(ids, root, proc, model, bs=16):
    preds = {}
    for i in tqdm(range(0, len(ids), bs), desc="Teacher"):
        batch_ids = ids[i:i+bs]
        imgs = [Image.open(f"{root}/{sid}_image.png").convert('RGB') for sid in batch_ids]
        inputs = proc(images=imgs, return_tensors="pt").to(DEVICE)
        with autocast('cuda', dtype=torch.float16):
            out = model(**inputs).predicted_depth
        for j, sid in enumerate(batch_ids):
            p = out[j:j+1].float()
            p = F.interpolate(p.unsqueeze(0), (448, 448), mode='bilinear', align_corners=False)
            p = p.squeeze().cpu().numpy()
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)
            preds[sid] = p.astype(np.float32)
    return preds

teacher_preds = gen_teacher(all_ids, CFG['train_root'], teacher_proc, teacher)
del teacher, teacher_proc; gc.collect(); torch.cuda.empty_cache()
print(f"Generated {len(teacher_preds)} predictions ✓")

Teacher: 100%|██████████| 575/575 [03:28<00:00,  2.75it/s]


Generated 9200 predictions ✓


---
## 3 · Dataset

In [9]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=10, p=0.6),
    A.OneOf([
        A.ColorJitter(0.25, 0.25, 0.25, 0.08),
        A.RandomBrightnessContrast(0.25, 0.25),
        A.HueSaturationValue(15, 25, 15),
    ], p=0.6),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.GaussNoise(var_limit=(10, 40)),
    ], p=0.15),
    A.CoarseDropout(max_holes=8, max_height=40, max_width=40, p=0.25),
], additional_targets={'depth': 'mask', 'teacher': 'mask'})

class DS(Dataset):
    def __init__(self, ids, root, t_preds, aug=None):
        self.ids, self.root, self.t_preds, self.aug = ids, root, t_preds, aug
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        sid = self.ids[i]
        img = cv2.cvtColor(cv2.imread(f"{self.root}/{sid}_image.png"), cv2.COLOR_BGR2RGB).astype(np.float32)/255
        dep = cv2.imread(f"{self.root}/{sid}_depth_map.png", 0).astype(np.float32)/255
        tea = self.t_preds[sid]
        if self.aug:
            r = self.aug(image=img, depth=dep, teacher=tea)
            img, dep, tea = r['image'], r['depth'], r['teacher']
        img = np.transpose((img - MEAN) / STD, (2,0,1)).astype(np.float32)
        return {'image': img, 'depth': dep, 'teacher': tea}

train_loader = DataLoader(DS(train_ids, CFG['train_root'], teacher_preds, train_aug),
                          batch_size=CFG['batch_size'], shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(DS(val_ids, CFG['train_root'], teacher_preds),
                        batch_size=8, shuffle=False, num_workers=2)
print(f"Train: {len(train_loader)} | Val: {len(val_loader)}")

Train: 287 | Val: 7


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_15140/2352752998.py:14: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 40)),
/tmp/ipykernel_15140/2352752998.py:16: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=40, max_width=40, p=0.25),


---
## 4 · Model

In [10]:
class TinyDepth(nn.Module):
    def __init__(self, enc='mobilenetv3_small_100', dec_ch=48, dropout=0.1):
        super().__init__()
        self.enc = timm.create_model(enc, pretrained=True, features_only=True, out_indices=(1,2,3,4))
        chs = self.enc.feature_info.channels()
        
        self.proj = nn.ModuleList([nn.Sequential(
            nn.Conv2d(c, dec_ch, 1, bias=False), nn.BatchNorm2d(dec_ch), nn.ReLU(True)
        ) for c in chs])
        
        self.up = nn.ModuleList([nn.Sequential(
            nn.Conv2d(dec_ch, dec_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(dec_ch), nn.ReLU(True)
        ) for _ in range(4)])
        
        self.head = nn.Sequential(
            nn.Conv2d(dec_ch, dec_ch, 3, 1, 1), nn.ReLU(True), nn.Dropout2d(dropout),
            nn.Conv2d(dec_ch, 1, 1), nn.Sigmoid()
        )
    
    def forward(self, x):
        feats = [p(f) for p, f in zip(self.proj, self.enc(x))]
        d = self.up[3](feats[3])
        d = F.interpolate(d, size=feats[2].shape[2:], mode='nearest') + feats[2]
        d = self.up[2](d)
        d = F.interpolate(d, size=feats[1].shape[2:], mode='nearest') + feats[1]
        d = self.up[1](d)
        d = F.interpolate(d, size=feats[0].shape[2:], mode='nearest') + feats[0]
        d = self.up[0](d)
        d = F.interpolate(d, size=(448, 448), mode='bilinear', align_corners=False)
        return self.head(d).squeeze(1)

model = TinyDepth(CFG['encoder_name'], CFG['decoder_channels'], CFG['dropout']).to(DEVICE)
print(f"Model: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

with torch.no_grad():
    assert model(torch.randn(2,3,448,448,device=DEVICE)).shape == (2,448,448)
print("Forward OK ✓")

Model: 1.06M params
Forward OK ✓


---
## 5 · Evaluation-Aligned Loss (KEY FIX!)

In [11]:
def per_sample_normalize(x):
    """Normalize each sample to [0, 1] range - matches evaluation!"""
    B = x.shape[0]
    x_flat = x.view(B, -1)
    x_min = x_flat.min(dim=1, keepdim=True)[0].view(B, 1, 1)
    x_max = x_flat.max(dim=1, keepdim=True)[0].view(B, 1, 1)
    return (x - x_min) / (x_max - x_min).clamp(min=1e-6)


class EvalAlignedLoss(nn.Module):
    """
    Loss that matches the evaluation metric exactly:
    1. Normalize pred and GT per-sample to [0, 1]
    2. Compute MSE (since eval uses RMSE)
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.w_d = cfg['w_distill_start']
    
    def set_progress(self, p):
        self.w_d = self.cfg['w_distill_start'] + (self.cfg['w_distill_end'] - self.cfg['w_distill_start']) * p
    
    def forward(self, pred, gt, teacher):
        # Normalize to match evaluation
        pred_norm = per_sample_normalize(pred)
        gt_norm = per_sample_normalize(gt)
        teacher_norm = per_sample_normalize(teacher)
        
        # MSE loss (matches RMSE metric)
        mse_loss = F.mse_loss(pred_norm, gt_norm)
        
        # Gradient loss (edge-aware)
        p_dx = pred_norm[:, :, :-1] - pred_norm[:, :, 1:]
        p_dy = pred_norm[:, :-1, :] - pred_norm[:, 1:, :]
        g_dx = gt_norm[:, :, :-1] - gt_norm[:, :, 1:]
        g_dy = gt_norm[:, :-1, :] - gt_norm[:, 1:, :]
        grad_loss = F.l1_loss(p_dx, g_dx) + F.l1_loss(p_dy, g_dy)
        
        # Multi-scale MSE
        ms_loss = 0
        for scale in [2, 4]:
            p_s = F.avg_pool2d(pred_norm.unsqueeze(1), scale).squeeze(1)
            g_s = F.avg_pool2d(gt_norm.unsqueeze(1), scale).squeeze(1)
            ms_loss += F.mse_loss(p_s, g_s)
        ms_loss /= 2
        
        # Distillation (also normalized)
        distill_loss = F.mse_loss(pred_norm, teacher_norm)
        
        # Total
        total = mse_loss + 0.2 * grad_loss + 0.3 * ms_loss + self.w_d * distill_loss
        
        return {
            'total': total,
            'mse': mse_loss,
            'grad': grad_loss,
            'ms': ms_loss,
            'distill': distill_loss,
            'w_d': self.w_d
        }

criterion = EvalAlignedLoss(CFG)
print("Evaluation-aligned loss defined ✓")

Evaluation-aligned loss defined ✓


---
## 6 · Training

In [12]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    mses = []
    for b in loader:
        p = model(b['image'].to(DEVICE)).float()
        g = b['depth'].to(DEVICE).float()
        p_n = per_sample_normalize(p)
        g_n = per_sample_normalize(g)
        mses.append(((p_n - g_n)**2).mean().item())
    return np.sqrt(np.mean(mses))

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])

total_steps = CFG['epochs'] * len(train_loader)
warmup_steps = CFG['warmup_epochs'] * len(train_loader)

def lr_fn(step):
    if step < warmup_steps: return step / warmup_steps
    return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)
scaler = GradScaler('cuda')

print(f"Training {CFG['epochs']} epochs, {total_steps} steps")

Training 100 epochs, 28700 steps


In [ ]:
best_rmse = 1.0
history = defaultdict(list)

for epoch in range(CFG['epochs']):
    model.train()
    criterion.set_progress(epoch / CFG['epochs'])
    losses = []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG['epochs']}")
    for batch in pbar:
        img = batch['image'].to(DEVICE, non_blocking=True)
        dep = batch['depth'].to(DEVICE, non_blocking=True)
        tea = batch['teacher'].to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            pred = model(img)
        
        loss = criterion(pred.float(), dep.float(), tea.float())
        
        if not (torch.isnan(loss['total']) or torch.isinf(loss['total'])):
            scaler.scale(loss['total']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(loss['total'].item())
        
        pbar.set_postfix({'loss': f"{loss['total'].item():.4f}", 'mse': f"{loss['mse'].item():.4f}"})
    
    history['loss'].append(np.mean(losses))
    
    # Validate every 5 epochs
    if (epoch + 1) % 5 == 0:
        val_rmse = evaluate(model, val_loader)
        history['val_rmse'].append(val_rmse)
        print(f"  Val RMSE: {val_rmse:.5f} | w_d: {criterion.w_d:.3f}")
        
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            torch.save(model.state_dict(), 'best.pt')
            print(f"  ★ New best!")

print(f"\n✓ Training complete. Best RMSE: {best_rmse:.5f}")

Epoch 5/100: 100%|██████████| 287/287 [00:49<00:00,  5.84it/s, loss=0.0422, mse=0.0257]


  Val RMSE: 0.11588 | w_d: 0.289
  ★ New best!


Epoch 10/100: 100%|██████████| 287/287 [00:54<00:00,  5.28it/s, loss=0.0170, mse=0.0097]


  Val RMSE: 0.09581 | w_d: 0.275
  ★ New best!


Epoch 15/100: 100%|██████████| 287/287 [00:51<00:00,  5.59it/s, loss=0.0113, mse=0.0062]


...

Epoch 40/100: 100%|██████████| 287/287 [00:45<00:00,  6.29it/s, loss=0.0062, mse=0.0032]


  Val RMSE: 0.05184 | w_d: 0.191
  ★ New best!


Epoch 45/100: 100%|██████████| 287/287 [00:45<00:00,  6.31it/s, loss=0.0070, mse=0.0038]


  Val RMSE: 0.05492 | w_d: 0.177


Epoch 48/100:  18%|█▊        | 51/287 [00:09<00:40,  5.81it/s, loss=0.0085, mse=0.0049]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['loss']); ax[0].set_title('Train Loss')
ax[1].plot(history['val_rmse']); ax[1].set_title('Val RMSE'); ax[1].axhline(0.135, c='r', ls='--', label='Previous best')
ax[1].legend()
plt.show()

---
## 7 · Export

In [ ]:
class ExportModel(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.base((x - self.mean) / self.std)

export_base = TinyDepth(CFG['encoder_name'], CFG['decoder_channels'], 0.0).to(DEVICE)
export_base.load_state_dict(torch.load('best.pt', weights_only=True))
export_base.eval().float()

export_model = ExportModel(export_base).cuda().float().eval()

dummy = torch.randn(8, 3, 448, 448, device='cuda', dtype=torch.float32)
with torch.no_grad():
    out = export_model(dummy)
    print(f"Output: {out.shape}, range [{out.min():.3f}, {out.max():.3f}]")

In [ ]:
torch.onnx.export(
    export_model, dummy, CFG['onnx_path'],
    export_params=True, opset_version=14, do_constant_folding=True,
    input_names=['input'], output_names=['output']
)

onnx.checker.check_model(onnx.load(CFG['onnx_path']))
size_mb = os.path.getsize(CFG['onnx_path']) / 1024**2
print(f"\n✓ ONNX: {size_mb:.2f} MB")

---
## 8 · Benchmark

In [ ]:
sess = ort.InferenceSession(CFG['onnx_path'], providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name

def infer(x): return sess.run(None, {inp_name: x.astype(np.float32)})[0]

out = infer(np.random.randn(8,3,448,448).astype(np.float32))
print(f"ONNX: {out.shape}, [{out.min():.3f}, {out.max():.3f}]")

In [ ]:
# Full validation RMSE
sq_err = []
for sid in tqdm(val_ids, desc="Val"):
    img = cv2.cvtColor(cv2.imread(f"{CFG['train_root']}/{sid}_image.png"), cv2.COLOR_BGR2RGB).astype(np.float32)/255
    img = np.transpose(img, (2,0,1))[None]
    gt = cv2.imread(f"{CFG['train_root']}/{sid}_depth_map.png", 0).astype(np.float32)/255
    
    batch = np.tile(img, (8,1,1,1))
    pred = infer(batch)[0]
    
    pred = (pred - pred.min()) / (pred.max() - pred.min() + 1e-8)
    gt = (gt - gt.min()) / (gt.max() - gt.min() + 1e-8)
    sq_err.append(((pred - gt)**2).mean())

rmse = np.sqrt(np.mean(sq_err))
print(f"\nONNX Val RMSE: {rmse:.5f}")

In [ ]:
# Speed
dummy = np.random.randn(8,3,448,448).astype(np.float32)
for _ in range(5): infer(dummy)
times = []
for _ in range(20):
    t0 = time.time()
    infer(dummy)
    times.append(time.time() - t0)

med = np.median(times)
acc = 4 / (4 + (10*rmse)**2)
size_s = (50 - size_mb) / 20
speed_s = 0.16 + np.log10(7 - (2/5.33)*med)

print(f"\n{'='*50}")
print(f"RMSE: {rmse:.5f} → Acc: {acc:.4f}")
print(f"Size: {size_mb:.2f} MB → {size_s:.4f}")
print(f"Time: {med:.4f}s → {speed_s:.4f}")
print(f"{'='*50}")
print(f"★ Score: {acc * size_s * speed_s:.4f}")

---
## 9 · Submit

In [ ]:
!du -h {CFG['onnx_path']}

In [ ]:
%%capture
!git clone --depth 1 https://github.com/chater-marzougui/MLS-GOAT.git
!mv MLS-GOAT/GOAT .
!rm -rf MLS-GOAT

In [ ]:
from GOAT import submit, leaderboard
submit.challenge2(CFG['onnx_path'], "DataC'EPT", "&G&MbH*cbgcgDatD")

In [ ]:
leaderboard.getLB_chall2("DataC'EPT")